In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer

In [ ]:
df = pd.read_csv(r"..\data\en.openfoodfacts.org.products.csv.gz", sep="\t", encoding='utf-8', nrows=10000)

In [ ]:
scores_connus = ['a', 'b', 'c', 'd', 'e']
df_connu = df[df['nutriscore_grade'].isin(scores_connus)].copy()

# DataFrame avec le reste (unknown, not-applicable, ou NaN éventuels)
df_inconnu = df[~df['nutriscore_grade'].isin(scores_connus)].copy()

# Vérification rapide
print("Connu :", df_connu.shape)
print("Inconnu :", df_inconnu.shape)
print(df_connu['nutriscore_grade'].value_counts())
print(df_inconnu['nutriscore_grade'].value_counts(dropna=False))

In [ ]:
from sklearn.model_selection import train_test_split

features = ['energy_100g', 'saturated-fat_100g', 'sugars_100g', 'salt_100g',
            'fiber_100g', 'proteins_100g', 'fruits-vegetables-legumes_100g',
            'fat_100g', 'carbohydrates_100g']

X = df_connu[features]  # Seules CES colonnes sont sélectionnées, tout le reste est ignoré
y = df_connu['nutriscore_grade']    

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#test_size should be between 0.0 and 1.0 and represent the proportion of the dataset to include in the test split.

In [ ]:
#Nettoyage minimal des données
percent_missing = df_connu[features].isnull().sum() * 100 / len(df_connu)
percent_missing.sort_values(ascending=False,inplace=True)

In [ ]:
## On retire les features avec + de 60% de valeurs manquantes

columns_to_drop = percent_missing[percent_missing.values > threshold].index
print(columns_to_drop)

df_restr_8 = df_connu[features + ['nutriscore_grade']].drop(columns=columns_to_drop)

print(df_restr_8.shape)
df_restr_8.head()

In [ ]:
##il y a des lignes avec des NaN dans chaque features, il faut les supprimer ! 

df_restr_8[df_restr_8.isna().sum(axis=1) >= 5]
(df_restr_8.isna().sum(axis=1) == 7).sum()

In [ ]:
from sklearn.impute import SimpleImputer

raw_df_restr_8 = df_restr_8.copy() ##je garde une copie du dataframe avant imputation pour pouvoir comparer les résultats après imputation.


# On sépare les features (à imputer) de la cible
features_restantes = df_restr_8.columns.drop('nutriscore_grade')

imputer = SimpleImputer(strategy='mean')

df_restr_8[features_restantes] = imputer.fit_transform(df_restr_8[features_restantes])

print(df_restr_8.isnull().sum())  # vérification : tout devrait être à 0
df_restr_8.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

cols = ['energy_100g', 'saturated-fat_100g', 'sugars_100g', 'salt_100g',
        'proteins_100g', 'fat_100g', 'carbohydrates_100g']

infos = df_connu.loc[df_restr_8.index, ['product_name', 'countries_en']]

fig, axs = plt.subplots(4, 2, figsize=(8, 16))

for ax, col in zip(axs.flat, cols):
    sns.boxplot(y=df_restr_8[col], ax=ax)
    ax.set_title(col)
    Q1 = df_restr_8[col].quantile(0.25)
    Q3 = df_restr_8[col].quantile(0.75)
    IQR = Q3 - Q1
    masque = (df_restr_8[col] < Q1 - 1.5*IQR) | (df_restr_8[col] > Q3 + 1.5*IQR)

    print(f"\n{col} : {masque.sum()} outliers selon Tukey")
    print(f"Bornes : [{Q1 - 1.5*IQR:.2f} ; {Q3 + 1.5*IQR:.2f}]")
    print(df_restr_8.loc[masque, [col]].join(infos)
          .sort_values(col, ascending=False).head(10).to_string())

axs.flat[-1].set_visible(False)
plt.tight_layout()
plt.show()